# Try a student prediction

Enter a **fictional or de-identified student** below and predict their final mathematics grade, G3 (0–20).

This notebook loads our selected **Random Forest regressors** and their fitted preprocessing. It does not train or tune anything. Run the cells from top to bottom. After editing the input, rerun the prediction cell.

- `USE_PRIOR_GRADES = False`: use the best model without G1/G2.
- `USE_PRIOR_GRADES = True`: use the later-stage model, only when both G1 and G2 are already known.

The result is a support signal for a teacher to review, not a guaranteed grade or a probability of failure. Use only information available at the time of prediction; total absences may not be known early in the course.

## 1. Load the saved models
Use the same Conda environment used for modelling. The loader checks saved model, data, preprocessing and library versions.

In [1]:
from pathlib import Path
import sys
import pandas as pd
from IPython.display import display

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "src/preprocessing.py").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / "src/preprocessing.py").exists():
    raise FileNotFoundError("Open this notebook from the EduPredict project or notebooks folder.")
if str(PROJECT_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / "src"))

from evaluation_utils import load_selected_models
from preprocessing import binary_mappings

dataset = pd.read_csv(PROJECT_ROOT / "data/student-mat.csv", sep=";")
models, metadata = load_selected_models(PROJECT_ROOT, dataset)
print("Both selected Random Forest models are ready.")

Both selected Random Forest models are ready.


## 2. Enter student details

Edit the values in the next cell. The example is fictional. Use the category codes exactly as shown.

| Input | Meaning / allowed values |
|---|---|
| school, sex, address | `GP` / `MS`; `F` / `M`; `U` urban / `R` rural |
| age | 15–22 years (dataset range) |
| famsize, Pstatus | `LE3` (three or fewer) / `GT3`; `T` together / `A` apart |
| Medu, Fedu | Mother's / father's education: 0 none, 1 primary, 2 grades 5–9, 3 secondary, 4 higher |
| Mjob, Fjob | `teacher`, `health`, `services`, `at_home`, `other` |
| reason | School choice: `home`, `reputation`, `course`, `other` |
| guardian | `mother`, `father`, `other` |
| traveltime | 1 <15 min, 2 15–30 min, 3 30–60 min, 4 >60 min |
| studytime | Weekly: 1 <2 h, 2 2–5 h, 3 5–10 h, 4 >10 h |
| failures | Previous failures: use observed dataset codes 0–3; source coding is ambiguous |
| schoolsup, famsup, paid | Extra school support, family support, paid subject classes: `yes` / `no` |
| activities, nursery, higher | Extracurricular activities, attended nursery, wants higher education: `yes` / `no` |
| internet, romantic | Home internet, romantic relationship: `yes` / `no` |
| famrel | Family relationship quality: 1 very bad – 5 excellent |
| freetime, goout | Free time / going out with friends: 1 very low – 5 very high |
| Dalc, Walc | Workday / weekend alcohol consumption: 1 very low – 5 very high |
| health | Health: 1 very bad – 5 very good |
| absences | Nonnegative integer; source documents 0–93, this dataset observes 0–75 |
| G1, G2 | First / second period grades: integers 0–20; required only in the later-stage mode |

Enter **raw values**: do not scale, encode or log-transform them. Do not enter G3; that is what we predict.

In [2]:
USE_PRIOR_GRADES = False

student = {
    "school": "GP",
    "sex": "F",
    "age": 17,
    "address": "U",
    "famsize": "GT3",
    "Pstatus": "T",
    "Medu": 3,
    "Fedu": 3,
    "Mjob": "services",
    "Fjob": "other",
    "reason": "course",
    "guardian": "mother",
    "traveltime": 1,
    "studytime": 2,
    "failures": 0,
    "schoolsup": "no",
    "famsup": "yes",
    "paid": "no",
    "activities": "yes",
    "nursery": "yes",
    "higher": "yes",
    "internet": "yes",
    "romantic": "no",
    "famrel": 4,
    "freetime": 3,
    "goout": 3,
    "Dalc": 1,
    "Walc": 1,
    "health": 4,
    "absences": 4,
}

# Used only when USE_PRIOR_GRADES is True. Replace with known grades.
prior_grades = {"G1": 12, "G2": 13}

## 3. Check the input and predict

This small check catches missing fields, spelling mistakes and invalid numeric codes before prediction. The saved pipeline handles all preprocessing automatically.

In [3]:
def make_student_input(student, prior_grades, use_prior_grades):
    scenario = "with_prior_grades" if use_prior_grades else "no_prior_grades"
    features = metadata[scenario]["input_features"]
    values = dict(student)
    if use_prior_grades:
        values.update(prior_grades)
    missing = set(features) - set(values)
    extra = set(values) - set(features)
    if missing or extra:
        raise ValueError(f"Check input fields. Missing: {sorted(missing)}; extra: {sorted(extra)}")

    categories = {name: set(mapping) for name, mapping in binary_mappings.items()}
    for name in ["Mjob", "Fjob"]:
        categories[name] = {"teacher", "health", "services", "at_home", "other"}
    categories.update(reason={"home", "reputation", "course", "other"},
                      guardian={"mother", "father", "other"})
    for name, allowed in categories.items():
        if values[name] not in allowed:
            raise ValueError(f"{name}: choose one of {sorted(allowed)}")

    ranges = {"age": (15, 22), "Medu": (0, 4), "Fedu": (0, 4),
              "traveltime": (1, 4), "studytime": (1, 4),
              "failures": (0, 3), "absences": (0, 93)}
    for name in ["famrel", "freetime", "goout", "Dalc", "Walc", "health"]:
        ranges[name] = (1, 5)
    if use_prior_grades:
        ranges.update(G1=(0, 20), G2=(0, 20))
    for name, (low, high) in ranges.items():
        value = values[name]
        if (isinstance(value, bool) or not isinstance(value, (int, float))
                or not low <= value <= high or not float(value).is_integer()):
            raise ValueError(f"{name}: enter a whole number from {low} to {high}.")
    return scenario, pd.DataFrame([values], columns=features)

scenario, student_input = make_student_input(student, prior_grades, USE_PRIOR_GRADES)
predicted_grade = float(models[scenario].predict(student_input)[0])
is_at_risk = predicted_grade < 10  # Apply threshold BEFORE rounding for display.

print("Model: Random Forest Regressor")
print("Inputs:", "with G1/G2" if USE_PRIOR_GRADES else "without G1/G2")
print(f"Predicted final grade (G3): {predicted_grade:.2f} / 20")
print("Risk flag:", "AT RISK — predicted G3 below 10" if is_at_risk
      else "NOT FLAGGED — predicted G3 is 10 or above")
if student["absences"] > 75:
    print("Note: absences exceeds the maximum observed in this dataset (75).")
display(student_input)

Model: Random Forest Regressor
Inputs: without G1/G2
Predicted final grade (G3): 11.19 / 20
Risk flag: NOT FLAGGED — predicted G3 is 10 or above


,school,sex,age,address,famsize,Pstatus,Medu,Fedu,Mjob,Fjob,reason,guardian,traveltime,studytime,failures,schoolsup,famsup,paid,activities,nursery,higher,internet,romantic,famrel,freetime,goout,Dalc,Walc,health,absences
0,GP,F,17,U,GT3,T,3,3,services,other,course,mother,1,2,0,no,yes,no,yes,yes,yes,yes,no,4,3,3,1,1,4,4


## What does the result mean?

The decimal grade is the model's estimate. A flag means the estimate is below our fixed threshold of 10. A student who is not flagged can still need support.

On the existing 79-student test set, the model without G1/G2 had MAE **3.10 grade points** and detected **10 of 27** at-risk students. The model with G1/G2 had MAE **1.17** and detected **27 of 27** in that sample. These are historical results, not a confidence interval or a guarantee for this input. See Notebook 04 for full evaluation, the G2 reference and limitations.

You cannot calculate accuracy or an error for a new student until their actual G3 is known. This notebook does not add your example to the training data or save the input. To try another student, edit section 2 and rerun sections 2 and 3.